# Object Tracking with Particle Filters in Python

Complete runnable companion: 5,000-particle color tracker, velocity prediction, image boundaries, BGR measurement errors, weighted resampling and Gaussian fuzzing. Run from a fresh Python3.11+ kernel. CPU only, no keys/GPU, no camera, GUI windows or interactive pauses. A checksum-verified7MB author-supplied walking video is downloaded from a pinned commit. All584 original1280×720 frames are processed in every demonstration; snapshots/animation only sample the display, not the filter computation.

The18 immutable original gists remain in a nonexecuting appendix. Repair their misspelled choice/mean calls, empty x-slice, missing video/frame-size setup and hidden global-particles argument. Keep the source's5,000 particles (not the older linked notebook's500), BGR target(66,63,105), velocity range0.5, position noise0.75 and velocity noise0.1. All powers1,2,8,16,32 are exercised. No target ground truth is supplied, so no tracking-accuracy percentage is invented.


In [ ]:
import subprocess
import sys
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--disable-pip-version-check",
               "numpy==2.3.3", "opencv-python-headless==4.13.0.92", "matplotlib==3.10.9",
               "pillow==12.1.1", "certifi==2026.10.23"], check=True)
import os
import io
import ssl
import hashlib
import urllib.request
from pathlib import Path
from time import perf_counter
import certifi
import numpy as np
import cv2
import matplotlib.pyplot as plt
from PIL import Image as PILImage
from IPython.display import display as notebook_display, Image as NotebookImage
cv2.setNumThreads(1)
np.set_printoptions(precision=6, suppress=True)
VIDEO_URL = "https://raw.githubusercontent.com/youssefHosni/Practical-Computer-Vision-In-Python/43aa5ff072e97b74d5a9296cb7b4a6c07afb16f5/Tracking%20Objects%20in%20Video%20with%20Particle%20Filters/walking.mp4"
VIDEO_SHA256 = "daaa87be86257169e91419a5c573cdada89edd7c041cb6d6c7f4a84d7f7543c8"
video_path = Path(os.environ.get("TDB_PARTICLE_VIDEO", "walking.mp4"))
if not video_path.exists():
    with urllib.request.urlopen(VIDEO_URL, context=ssl.create_default_context(cafile=certifi.where()), timeout=60) as response:
        video_bytes = response.read()
    assert hashlib.sha256(video_bytes).hexdigest() == VIDEO_SHA256
    video_path.write_bytes(video_bytes)
assert hashlib.sha256(video_path.read_bytes()).hexdigest() == VIDEO_SHA256
VFILENAME = str(video_path)
capture = cv2.VideoCapture(VFILENAME)
assert capture.isOpened(), "Cannot open the source video"
frame_width = int(capture.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT))
expected_frames = int(capture.get(cv2.CAP_PROP_FRAME_COUNT))
fps = capture.get(cv2.CAP_PROP_FPS)
ok, first_frame = capture.read()
capture.release()
assert ok
print("Python / OpenCV:", sys.version.split()[0], "/", cv2.__version__)
print("Video SHA-256:", VIDEO_SHA256)
print("Video:", frame_width, "x", frame_height, "; frames:", expected_frames, "; fps:", f"{fps:.6f}")

## Notebook playback support

The default path processes every frame without cv2.imshow/waitKey, which cannot run in Colab/headless kernels. Rendering happens only at checkpoints, as a notebook figure; it does not alter source frames used by the filter. Full GUI originals stay in the appendix. Capture release uses try/finally, even if a generator is closed early.


In [ ]:
rng = np.random.RandomState(0)
print("NumPy / OpenCV imports ready; source seed: 0.")

In [ ]:
capture = cv2.VideoCapture(VFILENAME)
assert capture.isOpened()
decoded_frames = 0
frame_digests = []
last_frame = None
try:
    while True:
        ok, frame = capture.read()
        if not ok:
            break
        assert frame.shape == (frame_height, frame_width, 3)
        frame_digests.append(hashlib.sha256(frame.tobytes()).hexdigest())
        decoded_frames += 1
        last_frame = frame
finally:
    capture.release()
assert decoded_frames == expected_frames
plt.figure(figsize=(10, 5))
plt.imshow(cv2.cvtColor(first_frame, cv2.COLOR_BGR2RGB))
plt.title("First frame of the original walking video")
plt.axis("off")
print("Decoded original video frames:", decoded_frames)
print("First / last frame SHA-256:", frame_digests[0], "/", frame_digests[-1])
plt.show()

In [ ]:
def get_frames(filename):
    video = cv2.VideoCapture(str(filename))
    if not video.isOpened():
        video.release()
        raise FileNotFoundError(f"Cannot open video: {filename}")
    try:
        while True:
            ok, frame = video.read()
            if not ok:
                break
            yield frame
    finally:
        video.release()
    yield None
generator_frames = sum(frame is not None for frame in get_frames(VFILENAME))
assert generator_frames == expected_frames
print("Generator frames:", generator_frames, "; capture released at completion.")

In [ ]:
def display(frame, particles=(), location=()):
    annotated = frame.copy()
    if len(particles):
        coordinates = np.asarray(particles[:, :2], dtype=int)
        valid = ((coordinates[:, 0] >= 0) & (coordinates[:, 0] < frame_width)
                 & (coordinates[:, 1] >= 0) & (coordinates[:, 1] < frame_height))
        for x, y in coordinates[valid]:
            cv2.circle(annotated, (int(x), int(y)), 1, (0, 255, 0), 1)
    if len(location):
        cv2.circle(annotated, tuple(map(int, location)), 15, (0, 0, 255), 5)
    return cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)
preview = display(first_frame, np.empty((0, 4)), (frame_width // 2, frame_height // 2))
assert np.array_equal(first_frame, cv2.cvtColor(display(first_frame), cv2.COLOR_RGB2BGR))
print("Notebook RGB preview:", preview.shape, "; original frame is not modified.")

In [ ]:
NUM_PARTICLES = 5000
VEL_RANGE = 0.5
def initialize_particles(random):
    particles = random.rand(NUM_PARTICLES, 4) * np.array([frame_width, frame_height, VEL_RANGE, VEL_RANGE])
    particles[:, 2:4] -= VEL_RANGE / 2
    return particles
particles = initialize_particles(np.random.RandomState(0))
print("Particle state shape:", particles.shape)
print("First three [x, y, vx, vy] states:")
print(particles[:3])

In [ ]:
results = {}
def run_demo(name, motion=False, bounded=False, weighted=False, noise=False, power=1, animate=False):
    random = np.random.RandomState(0)
    state = initialize_particles(random)
    positions = []
    snapshots = []
    animation_frames = []
    fallbacks = 0
    ess = []
    start = perf_counter()
    count = 0
    for frame in get_frames(VFILENAME):
        if frame is None:
            break
        if motion:
            state = apply_velocity(state)
        if bounded:
            state = enforce_edges(state)
        location = ()
        if weighted:
            errors = compute_errors(state, frame)
            weights = compute_weights(errors, state, power=power)
            probabilities, fallback = normalized_probabilities(weights, state)
            fallbacks += int(fallback)
            ess.append(1 / np.sum(probabilities ** 2))
            state, location, _ = resample(state, weights, random)
        if noise:
            state = enforce_edges(apply_noise(state, random))
        assert np.isfinite(state).all()
        if bounded:
            assert ((state[:, 0] >= 0) & (state[:, 0] <= frame_width - 1)).all()
            assert ((state[:, 1] >= 0) & (state[:, 1] <= frame_height - 1)).all()
        positions.append(state[:, :2].mean(axis=0))
        if count in [0, expected_frames // 2, expected_frames - 1]:
            snapshots.append((count, display(frame, state, location)))
        if animate and count % 8 == 0:
            animation_frames.append(PILImage.fromarray(display(frame, state, location)).resize((640, 360)))
        count += 1
    assert count == expected_frames
    result = dict(frames=count, particles=state.copy(), positions=np.array(positions), fallback_count=fallbacks, ess=np.array(ess))
    results[name] = result
    print(name, "| frames:", count, "| particles:", len(state), "| seconds:", f"{perf_counter() - start:.6f}")
    print("Final mean [x, y]:", result["positions"][-1])
    if weighted:
        print("Zero-weight fallbacks:", fallbacks, "| minimum ESS:", f"{min(ess):.6f}")
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for ax, (index, image) in zip(axes, snapshots):
        ax.imshow(image); ax.set_title(f"Frame {index}"); ax.axis("off")
    fig.suptitle(name); fig.tight_layout(); plt.show()
    if animation_frames:
        payload = io.BytesIO()
        animation_frames[0].save(payload, format="GIF", save_all=True, append_images=animation_frames[1:],
                                 duration=round(1000 * 8 / fps), loop=0)
        notebook_display(NotebookImage(data=payload.getvalue(), format="gif"))
        print("Rendered animation frames:", len(animation_frames))
    return result
print("Full-video demonstration runner ready; all stages use all584 frames.")

## Example 6: gistfile1.txt

In [ ]:
static_result = run_demo("Initialized static particles")

## Example 7: gistfile1.txt

In [ ]:
def apply_velocity(particles):
    particles[:, :2] += particles[:, 2:4]
    return particles
probe = particles[:3].copy()
print("Position increments:", (apply_velocity(probe) - particles[:3])[:, :2])

## Example 8: gistfile1.txt

In [ ]:
moving_result = run_demo("Velocity prediction before edge constraints", motion=True)

## Example 9: gistfile1.txt

In [ ]:
def enforce_edges(particles):
    particles[:, 0] = np.clip(particles[:, 0], 0, frame_width - 1)
    particles[:, 1] = np.clip(particles[:, 1], 0, frame_height - 1)
    return particles
print("Edge constraints: x in [0,", frame_width - 1, "]; y in [0,", frame_height - 1, "]")

## Example 10: gistfile1.txt

In [ ]:
bounded_result = run_demo("Velocity prediction with edge constraints", motion=True, bounded=True)

## Example 11: gistfile1.txt

In [ ]:
TARGET_COLOR = np.array([66, 63, 105], dtype=float)
def compute_errors(particles, frame):
    coordinates = particles[:, :2].astype(int)
    colors = frame[coordinates[:, 1], coordinates[:, 0]].astype(float)
    return np.sum((TARGET_COLOR - colors) ** 2, axis=1)
example_errors = compute_errors(enforce_edges(particles.copy()), first_frame)
print("Target BGR:", TARGET_COLOR.astype(int))
print("Squared color errors [minimum, maximum]:", example_errors.min(), example_errors.max())

## Example 12: gistfile1.txt

In [ ]:
def interior_mask(particles):
    return ((particles[:, 0] > 0) & (particles[:, 0] < frame_width - 1)
            & (particles[:, 1] > 0) & (particles[:, 1] < frame_height - 1))
def compute_weights(errors, particles, power=1):
    weights = np.max(errors) - errors
    weights[~interior_mask(particles)] = 0
    maximum = weights.max(initial=0)
    if maximum > 0:
        weights = (weights / maximum) ** power
    return weights
weights = compute_weights(example_errors, enforce_edges(particles.copy()))
print("Weight range:", weights.min(), weights.max())

## Example 13: gistfile1.txt

In [ ]:
def normalized_probabilities(weights, particles):
    weights = np.asarray(weights, dtype=float)
    if not np.isfinite(weights).all() or (weights < 0).any():
        raise ValueError("Weights must be finite and nonnegative")
    total = weights.sum()
    if total > 0:
        return weights / total, False
    eligible = interior_mask(particles).astype(float)
    if eligible.sum() == 0:
        eligible = np.ones(len(particles))
    return eligible / eligible.sum(), True
def resample(particles, weights, random):
    probabilities, _ = normalized_probabilities(weights, particles)
    indices = random.choice(len(particles), size=len(particles), p=probabilities)
    updated = particles[indices].copy()
    location = tuple(map(int, updated[:, :2].mean(axis=0)))
    return updated, location, indices
resampled, location, _ = resample(enforce_edges(particles.copy()), weights, np.random.RandomState(0))
print("Resampled particle state:", resampled.shape, "; mean location:", location)

## Example 14: gistfile1.txt

In [ ]:
with np.errstate(invalid="ignore"):
    legacy_probabilities = np.zeros(3) / np.zeros(3).sum()
try:
    np.random.RandomState(0).choice(3, size=3, p=legacy_probabilities)
except ValueError as error:
    print("Original zero-weight failure reproduced and caught:", str(error))
noiseless_result = run_demo("Resampling without Gaussian fuzzing", motion=True, bounded=True, weighted=True)

## Example 15: gistfile1.txt

In [ ]:
POS_SIGMA = 0.75
VEL_SIGMA = 0.1
def apply_noise(particles, random):
    noise = np.column_stack([random.normal(0, POS_SIGMA, len(particles)),
                             random.normal(0, POS_SIGMA, len(particles)),
                             random.normal(0, VEL_SIGMA, len(particles)),
                             random.normal(0, VEL_SIGMA, len(particles))])
    particles += noise
    return particles
noise_probe = apply_noise(np.zeros((NUM_PARTICLES, 4)), np.random.RandomState(0))
print("Actual noise standard deviations:", noise_probe.std(axis=0))

## Example 16: gistfile1.txt

In [ ]:
noise_result = run_demo("Gaussian fuzzing, power1", motion=True, bounded=True, weighted=True, noise=True, power=1)

## Example 17: gistfile1.txt

In [ ]:
power2_result = run_demo("Gaussian fuzzing, power2", motion=True, bounded=True, weighted=True, noise=True, power=2)

## Example 18: gistfile1.txt

In [ ]:
power8_result = run_demo("Gaussian fuzzing, power8", motion=True, bounded=True, weighted=True, noise=True, power=8)

## Remaining source-discussed powers:16 and32

In [ ]:
power16_result = run_demo("Gaussian fuzzing, power16", motion=True, bounded=True, weighted=True, noise=True, power=16, animate=True)
power32_result = run_demo("Gaussian fuzzing, power32", motion=True, bounded=True, weighted=True, noise=True, power=32)

## Independent regressions

In [ ]:
assert expected_frames == 584 and (frame_width, frame_height) == (1280, 720)
assert len(results) == 9 and all(v["frames"] == 584 and v["particles"].shape == (5000, 4) for v in results.values())
assert static_result["positions"].shape == (584, 2)
np.testing.assert_allclose(static_result["positions"], np.broadcast_to(particles[:, :2].mean(axis=0), (584, 2)), atol=1e-9)
np.testing.assert_allclose(moving_result["particles"][:, :2], particles[:, :2] + 584 * particles[:, 2:4], atol=1e-8)
# Independent scalar measurement / boundary / powers on all5000 states.
bounded = enforce_edges(particles.copy())
manual_errors = []
for x, y in bounded[:, :2].astype(int):
    color = first_frame[y, x]
    manual_errors.append(sum((float(TARGET_COLOR[c]) - int(color[c])) ** 2 for c in range(3)))
np.testing.assert_array_equal(example_errors, manual_errors)
edge_probe = np.array([[-5, 800, 0, 0], [1500, -1, 0, 0]], dtype=float)
np.testing.assert_array_equal(enforce_edges(edge_probe)[:, :2], [[0, 719], [1279, 0]])
for power in [1, 2, 8, 16, 32]:
    raw = np.max(example_errors) - np.array(manual_errors)
    raw[~interior_mask(bounded)] = 0
    manual = np.array([float(value) ** power for value in raw]); manual /= manual.sum()
    actual, _ = normalized_probabilities(compute_weights(example_errors, bounded, power), bounded)
    np.testing.assert_allclose(actual, manual, rtol=1e-12, atol=1e-15)
# Choice indices reconstructed independently from uniforms and cumulative probabilities.
probabilities, _ = normalized_probabilities(weights, bounded)
reference_random = np.random.RandomState(0)
uniforms = reference_random.random_sample(NUM_PARTICLES)
cdf = np.cumsum(probabilities); cdf /= cdf[-1]
manual_indices = np.searchsorted(cdf, uniforms, side="right")
actual_particles, actual_location, actual_indices = resample(bounded, weights, np.random.RandomState(0))
np.testing.assert_array_equal(actual_indices, manual_indices)
np.testing.assert_array_equal(actual_particles, bounded[manual_indices])
np.testing.assert_array_equal(actual_location, bounded[manual_indices, :2].mean(axis=0).astype(int))
zero_probabilities, fallback = normalized_probabilities(np.zeros(5000), bounded)
assert fallback and np.isfinite(zero_probabilities).all() and np.isclose(zero_probabilities.sum(), 1)
same_pixel = np.tile([100, 100, 0, 0], (5000, 1)).astype(float)
equal_errors = compute_errors(same_pixel, first_frame)
safe_weights = compute_weights(equal_errors, same_pixel, power=32)
safe_particles, safe_location, _ = resample(same_pixel, safe_weights, np.random.RandomState(0))
assert safe_location == (100, 100) and np.isfinite(safe_particles).all()
# Replay the complete final tracker independently in scalar formulas on all584 frames.
reference_random = np.random.RandomState(0)
reference = reference_random.rand(5000, 4) * np.array([1280, 720, 0.5, 0.5]); reference[:, 2:] -= 0.25
reference_positions = []
for frame in get_frames(VFILENAME):
    if frame is None:break
    reference[:, 0] += reference[:, 2]; reference[:, 1] += reference[:, 3]
    reference[:, 0] = np.maximum(0, np.minimum(1279, reference[:, 0])); reference[:, 1] = np.maximum(0, np.minimum(719, reference[:, 1]))
    coordinates = reference[:, :2].astype(int); colors = frame[coordinates[:, 1], coordinates[:, 0]].astype(float)
    errors = (66-colors[:, 0])**2 + (63-colors[:, 1])**2 + (105-colors[:, 2])**2
    raw = errors.max() - errors
    mask = (reference[:, 0]>0)&(reference[:, 0]<1279)&(reference[:, 1]>0)&(reference[:, 1]<719)
    raw[~mask] = 0
    if raw.max()>0:
        raw = (raw/raw.max())**16
    if raw.sum()>0:probs = raw/raw.sum()
    else:
        probs = mask.astype(float) if mask.any() else np.ones(5000); probs /= probs.sum()
    cumulative = np.cumsum(probs); cumulative /= cumulative[-1]
    indices = np.searchsorted(cumulative, reference_random.random_sample(5000), side="right")
    reference = reference[indices].copy()
    for column, sigma in enumerate([0.75, 0.75, 0.1, 0.1]):reference[:, column] += reference_random.normal(0, sigma, 5000)
    reference[:, 0] = np.maximum(0, np.minimum(1279, reference[:, 0])); reference[:, 1] = np.maximum(0, np.minimum(719, reference[:, 1]))
    reference_positions.append(reference[:, :2].mean(axis=0))
np.testing.assert_allclose(reference_positions, power16_result["positions"], rtol=1e-10, atol=1e-8)
np.testing.assert_allclose(reference, power16_result["particles"], rtol=1e-10, atol=1e-8)
print("All18 recovered examples and both additional power variants run on all584 original frames with5000 particles.")
print("Independent5000-state color errors, all five weight powers, edge bounds, inverse-CDF resampling and zero-weight recovery pass.")
print("Independent complete584-frame power16 replay matches every trajectory mean and all5000 final particle states.")
print("No ground-truth annotations supplied: no tracking-accuracy metric is claimed.")

## Maintenance and interpretation

The article source has18gists; the linked older notebook uses500 particles and an undefined Man.mp4 path. This companion follows the article's5000 and actualwalking.mp4; frame dimensions are discovered, not assumed. The prose reverses target BGR channels (105,63,66); keep actual source code(66,63,105). The source resample gist misspells np.random.choice/np.mean and uses an empty x-slice. Fix these; make state arguments explicit.

Vectorized velocity/clip/color/weight operations preserve scalar formulas and are independently checked. Normalize weights before exponentiation (identical relative probabilities); handle all-zero weights with uniform eligible interior particles, or all particles if none are interior. Noise uses the actual source0.75/0.1 values (not prose's1/0.5), source seed0 and independent per-demo replay. Clamp after fuzzing for safe state/display. The original collapse error is deliberately reproduced and caught as teaching content, not allowed to crash the runnable notebook.

This is a color-matching demonstration, not a robust general-purpose identity tracker, Bayesian likelihood calibration, accuracy benchmark or universal guarantee that more particles/higher powers improve tracking. Similar background/clothing colors, illumination and target-color ambiguities matter. A red mean circle is an estimate, not annotated truth. The source's simplistic Kalman-filter comparison and function-optimization description are not a complete probabilistic definition. These findings stay in notebook/private review; original website prose is retained.


## Immutable original gist appendix (not executable)

### b8cd13ddd702b1cc4cf283e6fe9c8fdf / 782fdf083a452feb99f0c6a6e964cb19ef62e2a7

Original filename: gistfile1.txt

```python
import numpy as np
import cv2

# Repeatability
np.random.seed(0)
```

### fb5fbcc6709cf01666d61ad2569d1254 / 51b9ab97253b0fe684e173048c025c1d69c59dac

Original filename: gistfile1.txt

```python
import cv2
import numpy as np


# Create a VideoCapture object and read from input file
# If the input is the camera, pass 0 instead of the video file name
cap = cv2.VideoCapture('walking.mp4')

# Check if camera opened successfully
if (cap.isOpened()== False): 
  print("Error opening video stream or file")
 
# Read until video is completed
while(cap.isOpened()):
  # Capture frame-by-frame
  ret, frame = cap.read()

  if ret == True:
    # Display the resulting frame
    cv2.imshow('Frame',frame)
 
    # Press Q on keyboard to  exit
    if cv2.waitKey(30) & 0xFF == ord('q'):
      break
 
  # Break the loop
  else: 
    break
 
# When everything done, release the video capture object
cap.release()
 
# Closes all the frames
cv2.destroyAllWindows()
```

### 20ce954626d8573bdfa786c27daad579 / 982cb56fe31de0a037f72b4bbc3691de46dbcea8

Original filename: gistfile1.txt

```python
def get_frames(filename):
    video = cv2.VideoCapture(filename)
    while video.isOpened():
        ret, frame =video.read()
        if ret:
            yield frame 
        else:
            break
    video.release()        
    yield None
```

### d98751c5a0efd1826431fc3e3956e607 / 4190c0fdcd0992769aacbbb1f999fe0d578ba13d

Original filename: gistfile1.txt

```python
def display(frame, particles, location):
    if len(particles)> 0:
        for i in range(NUM_PARTICLES):
            x = int(particles[i,0])
            y = int(particles[i,1])
            cv2.circle(frame,(x,y),1,(0,255,0),1)
    if len(location) > 0:
        cv2.circle(frame,location,15,(0,0,255),5)
    cv2.imshow('frame', frame)
    #stop the video if pressing the escape button
    if cv2.waitKey(30)==27:
        if cv2.waitKey(0)==27:
            return True 

    return False
```

### 6e4e50c0ca79f8fdfff35784f9852220 / 00f50a4c1084746fdcdcf40b3ed693cd08cfd465

Original filename: gistfile1.txt

```python
NUM_PARTICLES = 5000
VEL_RANGE = 0.5

def initialize_particles():
    particles = np.random.rand(NUM_PARTICLES,4)
    particles = particles * np.array((frame_width,frame_height, VEL_RANGE,VEL_RANGE))
    particles[:,2:4] -= VEL_RANGE/2.0
    return particles
```

### 390f391c73e9d7d4971a6fecbb786311 / 15f3f8a58424dc85fb13ef121f2d1a2237b56dec

Original filename: gistfile1.txt

```python
location =[]
particles = initialize_particles()


for frame in get_frames(VFILENAME):
    if frame is None: break
    terminate = display(frame, particles, location)
    if terminate:
        break

cv2.destroyAllWindows()
```

### 3735d5c458a8271f9c6aa73efa271403 / cb084edbdcc4940d981ac33a94f025f0601b4be4

Original filename: gistfile1.txt

```python
def apply_velocity(particles):
    particles[:,0] += particles[:,2]
    particles[:,1] += particles[:,3]

    return particles
```

### a3ca433edaa1462b470aaf7b5e8b57f7 / 71a30341fbc63e392bdb3e22994e49ed67928451

Original filename: gistfile1.txt

```python
location = []
particles = initialize_particles()


for frame in get_frames(VFILENAME):
    if frame is None: break
    particles = apply_velocity(particles)

    terminate = display(frame, particles, location)
    if terminate:
        break

cv2.destroyAllWindows()
```

### 25913e264a7f957ba9298624dce22f9c / 6ff8121f5b4bd9bd548a25588845ff053152661c

Original filename: gistfile1.txt

```python
def enforce_edges(particles):
    for i in range(NUM_PARTICLES):
        particles[i,0] = max(0,min(frame_width-1, particles[i,0]))
        particles[i,1] = max(0,min(frame_height-1, particles[i,1]))
    return particles
```

### 8ff6d36588b23848344a2ce1bbbf2bbe / 2103e0f329b4e1a15e42f65b805a1e0d186e322a

Original filename: gistfile1.txt

```python
location = []
particles = initialize_particles()


for frame in get_frames(VFILENAME):
    if frame is None: break
    particles = apply_velocity(particles)
    particles = enforce_edges(particles)
    terminate = display(frame, particles, location)
    if terminate:
        break

cv2.destroyAllWindows()
```

### 332d531519a590a8a125bda579ec522c / ab18919da26490824864c83aaa30717181a09047

Original filename: gistfile1.txt

```python
TARGET_COLOR = np.array((66,63, 105))

def compute_errors(particles, frame):
    
    errors = np.zeros(NUM_PARTICLES)
    for i in range(NUM_PARTICLES):
        x = int(particles[i,0])
        y= int(particles[i,1])
        pixel_color = frame[y, x, :]
        errors[i] = np.sum((TARGET_COLOR - pixel_color)**2)
    
    return errors
```

### 151519ebef943b86383a648673fe22cc / bf37d63b2a4c5c8a3af807f1cca1b0f7b028c003

Original filename: gistfile1.txt

```python
def compute_weights(errors):
    weights = np.max(errors) - errors
    
    weights[
        (particles[:,0]==0) |
        (particles[:,0]==frame_width-1) |
        (particles[:,1]==0) |
        (particles[:,1]==frame_height-1) ] = 0  
        
    return weights
```

### f6b4f5dddd8f3fd00ef55fb7b23127d0 / cf496ad16b005237c05860e9ba66d791821daa86

Original filename: gistfile1.txt

```python
def resample(particles, weights):
    probabilities = weights / np.sum(weights)
    index_numbers = np.random.choicse(
        NUM_PARTICLES,
        size=NUM_PARTICLES,
        p=probabilities)
    particles = particles[index_numbers, :]
    
    x = np.mean(particles[:0])
    y= np.nean(particles[:,1])
    
    return particles, [int(x), int(y)]
```

### 328f7070c8137c6095d62a4d78f7afcd / 52b9c1bed78dad1cd9840ab00db7cc3303ea461b

Original filename: gistfile1.txt

```python
particles = initialize_particles()


for frame in get_frames(VFILENAME):
    if frame is None: break
    particles = apply_velocity(particles)
    particles = enforce_edges(particles)
    errors = compute_errors(particles, frame)
    weights = compute_weights(errors)
    particles, location = resample(particles, weights)

    terminate = display(frame, particles, location)
    if terminate:
        break

cv2.destroyAllWindows()
```

### 296900d0c7e1ccad7b19b5a872f4bbe5 / 30687a584382a320dd104215d7ae2c15084dd555

Original filename: gistfile1.txt

```python
POS_SIGMA = 0.75
VEL_SIGMA = 0.1

def apply_noise(particles):
    noise= np.concatenate(
    (
        np.random.normal(0.0, POS_SIGMA, (NUM_PARTICLES,1)),
        np.random.normal(0.0, POS_SIGMA, (NUM_PARTICLES,1)),
        np.random.normal(0.0, VEL_SIGMA, (NUM_PARTICLES,1)),
        np.random.normal(0.0, VEL_SIGMA, (NUM_PARTICLES,1))
    
    ),
    axis=1)
    
    particles += noise
    return particles
```

### 410cd97a2ba67908590d4754c065e8cb / b81f224b172f54f8ec6c8b3bc48ff140e95dc288

Original filename: gistfile1.txt

```python
particles = initialize_particles()

for frame in get_frames(VFILENAME):
    if frame is None: break

    particles = apply_velocity(particles)
    particles = enforce_edges(particles)
    errors = compute_errors(particles, frame)
    weights = compute_weights(errors)
    particles, location = resample(particles, weights)
    particles = apply_noise(particles)
    terminate = display(frame, particles, location)
    if terminate:
        break
cv2.destroyAllWindows()
```

### 7ceabc4bdcc7166ed08bb4f7b9024e91 / 35b7d800c92d4865b2d3aba9edeb88ada3d1fd8b

Original filename: gistfile1.txt

```python
def compute_weights(errors):
    weights = np.max(errors) - errors
    
    weights[
        (particles[:,0]==0) |
        (particles[:,0]==frame_width-1) |
        (particles[:,1]==0) |
        (particles[:,1]==frame_height-1) ] = 0  
    
    weights = weights**2
    
    return weights
```

### 24e24e6c921a776b759a1a6caf07bcd3 / a8f97a7946dc6ad705e51ef226e868c9cff1497d

Original filename: gistfile1.txt

```python
def compute_weights(errors):
    weights = np.max(errors) - errors
    
    weights[
        (particles[:,0]==0) |
        (particles[:,0]==frame_width-1) |
        (particles[:,1]==0) |
        (particles[:,1]==frame_height-1) ] = 0  
    
    weights = weights**8
    
    return weights
```

